In [1]:
import osmnx as ox
import pandas as pd
import numpy as np
import networkx as nx

In [2]:
G = ox.graph_from_place("Raleigh, North Carolina, USA", network_type="drive")

In [3]:
print(ox.basic_stats(G))

{'n': 16502, 'm': 39220, 'k_avg': 4.753363228699552, 'edge_length_total': 5057625.23490873, 'edge_length_avg': 128.9552584117473, 'streets_per_node_avg': 2.662404557023391, 'streets_per_node_counts': {0: 0, 1: 3959, 2: 168, 3: 9927, 4: 2382, 5: 65, 6: 1}, 'streets_per_node_proportions': {0: 0.0, 1: 0.2399103139013453, 2: 0.010180584171615562, 3: 0.6015634468549267, 4: 0.14434613986183492, 5: 0.003938916494970307, 6: 6.0598715307235486e-05}, 'intersection_count': 12543, 'street_length_total': 2870593.5437974166, 'street_segment_count': 21860, 'street_length_avg': 131.3171794966796, 'circuity_avg': 1.0750420592154168, 'self_loop_proportion': 0.010064043915827997}


In [4]:
df = pd.read_csv('../data/depot_dataset.csv')
node_ids = ox.nearest_nodes(G, df['longitude'].values, df['latitude'].values)
df['node_id'] = node_ids

In [5]:
depot = df.iloc[[0]]
stops = df.iloc[1:].sample(n=20, random_state=42)
sample = pd.concat([depot,stops]).reset_index(drop=True)
sample.to_csv('../data/sample_stops.csv')

In [6]:
node_list = sample['node_id'].to_list()
n = len(node_list)
distance_matrix = np.zeros((n,n))
for i, orig in enumerate(node_list):
    for j, dest in enumerate(node_list):
        if i != j:
            try:
                distance_matrix[i][j] = nx.shortest_path_length(
                    G, orig, dest, weight='length'
                )
            except:
                distance_matrix[i][j] = 999999999
print(distance_matrix)

[[    0.         13298.95360759 18739.10344063 28276.43492381
   6976.6601964  15206.71175046 18036.75850583 13517.63646874
  23273.31455849  8898.59824497  9006.9588399  13963.06166958
  15418.39467842 13533.00417098  9154.57658046 19135.76540405
  15269.44992648 13699.80607346  5746.74195774 15170.07964347
   9401.75291148]
 [13177.53755573     0.          6178.08908542 19465.88013379
   7121.588068   13625.70930379 17554.23169288   495.7773908
  14462.75976847 11862.26964539 12763.28874491  1808.92824698
  14935.86786547 14528.6133043   5854.41707627 18232.92443374
   9818.78308001  8249.13922699  8362.5018715  13911.61947015
   6642.45644675]
 [19178.42007212  6178.08908542     0.         14575.8979944
  12877.01603245 12742.76092601 16037.10299024  5965.02710751
   9572.77762909 14819.49490463 15775.94895851  6968.200985
  14358.84834188 15741.09250105 11411.46012622 16576.20093044
   7507.69838678  6373.75064895 14117.92983595 13028.67109237
  12820.54553217]
 [29331.96413369 196

In [7]:
np.save('../data/distance_matrix.npy', distance_matrix)
sample.to_csv('../data/sample_stops.csv', index=False)